In [1]:
%load_ext autoreload
%autoreload 2


In [3]:
import numpy as np
import matplotlib.pyplot as plt

from snudd.geometry import SolarAngles
from snudd.nsi import earth_matter, nsi_probabilities
from snudd.models import GeneralNSI, SM

from snudd.targets import nucleus_xe

In [4]:
nsi_model = GeneralNSI([[1, 0, 0],
                        [0, 1, 0],
                        [0, 0, 0]],
                        np.pi/4, 0)

nucleus_xe.update_model(nsi_model)


In [25]:
%%timeit
nucleus_xe.prepare_density()

114 ms ± 1.55 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [62]:
E_nus = np.geomspace(4e-3,1e1, 500) / 1e3

In [63]:
%%timeit
density_true = nucleus_xe._spec.density_calc.density(E_nus, '8B')


12.7 ms ± 208 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [61]:
%%timeit
density_interp = nucleus_xe._spec.nu_density_elements['8B'](E_nus)

133 μs ± 3.5 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [59]:
density_interp.shape

(12, 400)

## Testing earth evolution

In [5]:
t0 = 91
T  = 182*2

GranSasso = SolarAngles(latitude=42.47, t0=t0, T=T)
angles, weights = GranSasso.zenith_hist(bins=25)

In [7]:
DensityCalc = nsi_probabilities.DensityMatrixEarthCalculator(nsi_model, cetas=angles)

interpolated_rhos = DensityCalc.interpolate_earth_density_elements(E_nu_min=3.4640e-3, E_nu_max=1.8784e1)

/Users/foldenauer/local/opt/SNuDD/snudd/nsi/earth_matter.py:133: RuntimeWarning: invalid value encountered in arccos
  if np.arccos(ceta) >= np.pi/2:


In [17]:
print(angles/np.pi)

E_nus = np.geomspace(3.4640e-3, 1.8784e1, 200) / 1e3
print(interpolated_rhos['8B'][0](E_nus))

[0.10572381 0.12149488 0.13726594 0.15303701 0.16880808 0.18457915
 0.20035022 0.21612128 0.23189235 0.24766342 0.26343449 0.27920555
 0.29497662 0.31074769 0.32651876 0.34228983 0.35806089 0.37383196
 0.38960303 0.4053741  0.42114516 0.43691623 0.4526873  0.46845837
 0.48422944 0.5000005  0.51577157 0.53154264 0.54731371 0.56308477
 0.57885584 0.59462691 0.61039798 0.62616905 0.64194011 0.65771118
 0.67348225 0.68925332 0.70502438 0.72079545 0.73656652 0.75233759
 0.76810866 0.78387972 0.79965079 0.81542186 0.83119293 0.84696399
 0.86273506 0.87850613 0.8942772 ]
[[ 5.49214150e-01  5.49209133e-01  5.49201409e-01 ...  3.34982698e-01
   2.97997582e-01  3.48978904e-01]
 [ 4.06614977e-19 -2.23097084e-18  5.17867847e-17 ...  1.59701050e-17
   2.48397398e-18  3.15167672e-17]
 [ 5.12985157e-02  5.12837180e-02  5.12685681e-02 ... -3.06697494e-01
  -2.78758946e-01 -2.83386043e-01]
 ...
 [ 1.36611178e-02  1.36594358e-02  1.36579361e-02 ... -3.33262310e-02
  -3.69780139e-02 -4.24315905e-02]
 [ 2